# Stage 3 — Catheter / guidewire tracking (Colab/Kaggle BUILD)

**Thin notebook** — logic in `src/*`. YOLO11n (2 classes: catheter, guidewire) on **CathAction**, then **ByteTrack** for continuity. ByteTrack is Python around the detector, not inside the CoreML graph.

Acceptance = **fps + ID-switches**, not just IoU.

**Dataset:** CathAction `airvlab/CathAction` (HF). This notebook auto-pulls **`segmentation_human_train.zip`** (143 MB, human catheter+guidewire masks) — the right split for a clinical detector. The 41.8 GB action + 4.35 GB collision zips are other tasks; skip them.

In [ ]:
# 0) Repo + env. Colab mounts Drive so repo+runs persist; Kaggle uses /kaggle/working.
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/intv-img/interventional-imaging-pipeline'
except Exception:
    REPO = '/kaggle/working/interventional-imaging-pipeline'
REPO_URL = 'https://github.com/jugalmodi0111/interventional-imaging-pipeline.git'
if not os.path.exists(REPO):
    !git clone {REPO_URL} {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install "ultralytics>=8.2" pycocotools opencv-python pyyaml huggingface_hub
from src import env
E = env.setup()
assert E['device'] == 'cuda', 'Switch runtime to GPU.'
import torch
print('catheter build env ready | device', E['device'], '|', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

In [ ]:
# 1) Get CathAction into data/raw/cathaction.
#    Priority: Kaggle-attached copy -> else download the 143 MB human seg split from HF.
import os, glob, zipfile
os.makedirs('data/raw', exist_ok=True)
kag = [d for d in glob.glob('/kaggle/input/**/*athAction*', recursive=True)
             + glob.glob('/kaggle/input/**/*cathaction*', recursive=True) if os.path.isdir(d)]
if kag and not os.path.exists('data/raw/cathaction'):
    os.symlink(kag[0], 'data/raw/cathaction'); print('using Kaggle input:', kag[0])
os.makedirs('data/raw/cathaction', exist_ok=True)
if not glob.glob('data/raw/cathaction/**/*.*', recursive=True):
    from huggingface_hub import hf_hub_download
    zp = hf_hub_download(repo_id='airvlab/CathAction', repo_type='dataset',
                         filename='segmentation_human_train.zip')   # 143 MB, human catheter+guidewire masks
    with zipfile.ZipFile(zp) as z: z.extractall('data/raw/cathaction')
    print('downloaded + unzipped ->', zp)
# want more data? also fetch filename='segmentation_animal_phantom.zip' (9.88 GB) the same way.
n = len(glob.glob('data/raw/cathaction/**/*.*', recursive=True))
print('CathAction files:', n)
assert n, 'CathAction empty — check HF download / Kaggle attach'

In [ ]:
# 2) CathAction -> YOLO (catheter, guidewire). Importable converter (COCO or mask fallback).
import yaml
from src.data_prep import cathaction_to_yolo
cfg = yaml.safe_load(open('configs/catheter_track.yaml'))
cathaction_to_yolo.main(cfg)

In [ ]:
# 3) Train the 2-class detector on GPU (reuses the generalized train_detector).
from src.train.train_detector import train
best = train(cfg, project=f"{E['runs']}/catheter", device=0)  # device=0 = force GPU, fail loud if none
print('best ->', best)

In [ ]:
# 3b) (Optional) Build a real sample_clip.mp4 from one frame sequence, so tracking runs on an
#     actual video file. Skip if you attached video_action_understanding.zip (has procedure mp4s).
import glob, os, cv2
from collections import Counter
OUT_MP4 = 'data/raw/cathaction/sample_clip.mp4'
IMG = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
BAD = ('mask', 'guidewire', 'label', 'annotation', 'gt', '/seg')
have_mp4 = os.path.exists(OUT_MP4) or bool(glob.glob('data/raw/cathaction/**/*.mp4', recursive=True))
if not have_mp4:
    fr = [p for p in glob.glob('data/raw/cathaction/**/*', recursive=True) if p.lower().endswith(IMG)]
    fr = [p for p in fr if not any(b in p.lower().split('cathaction/', 1)[-1] for b in BAD)]
    if fr:
        d   = Counter(os.path.dirname(p) for p in fr).most_common(1)[0][0]   # biggest dir = one run
        seq = sorted(p for p in fr if os.path.dirname(p) == d)
        h, w = cv2.imread(seq[0]).shape[:2]
        vw = cv2.VideoWriter(OUT_MP4, cv2.VideoWriter_fourcc(*'mp4v'), 15, (w, h))
        for p in seq:
            im = cv2.imread(p)
            vw.write(im if im.shape[:2] == (h, w) else cv2.resize(im, (w, h)))
        vw.release()
        print(f'built {OUT_MP4} from {len(seq)} frames @ {d}')
    else:
        print('no frames found to build mp4 (cell 4 will fall back to frame-dir tracking)')
else:
    print('mp4 already present -> skip build')

In [ ]:
# 4) Track with ByteTrack. The seg split has NO video, so run on a frame SEQUENCE folder:
#    Ultralytics accepts an image dir and processes frames in sorted (=temporal) order for ID continuity.
import glob, os
from collections import Counter
from src.serve.track import track_yolo
IMG = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
BAD = ('mask', 'guidewire', 'label', 'annotation', 'gt', '/seg')   # skip label/mask dirs (not raw frames)

clip = 'data/raw/cathaction/sample_clip.mp4'
if not os.path.exists(clip):
    mp4s = glob.glob('data/raw/cathaction/**/*.mp4', recursive=True)
    clip = mp4s[0] if mp4s else None
if clip is None:
    frames = [p for p in glob.glob('data/raw/cathaction/**/*', recursive=True)
              if p.lower().endswith(IMG)]
    frames = [p for p in frames
              if not any(b in p.lower().split('cathaction/', 1)[-1] for b in BAD)]
    if frames:                                    # pick the most-populated dir = one imaging run
        clip = Counter(os.path.dirname(p) for p in frames).most_common(1)[0][0]
        k = sum(1 for p in frames if os.path.dirname(p) == clip)
        print(f'no mp4 -> tracking on frame dir: {clip} ({k} frames). Verify it is one raw sequence.')
if clip:
    track_yolo(best, source=clip, out=f"{E['runs']}/catheter/track.mp4", device=0)
else:
    print('no frames/mp4 under data/raw/cathaction — skipping ByteTrack demo (best.pt still exported)')

## Handoff
Copy `best.pt` to the Mac: `make export-coreml-yolo MODEL=best.pt` → `.mlpackage`, then per-frame edge detection via `src.serve.realtime --task det`; ByteTrack stays in the app loop (`src.serve.track`).

**Persist (Kaggle):** `/kaggle/working` is wiped on session death — use *Save & Run All (Commit)* or download `best.pt` before closing. Colab repo lives on Drive → runs persist.